# 03 — Classical Segmentation (L3 Baseline)

Course-aligned (Lecture 2 material). These methods serve as **interpretable baselines** — they prove we understand the fundamentals before jumping to deep learning.

Methods:
1. Otsu thresholding (global + multi-level)
2. Region growing (seed-based)
3. Watershed segmentation
4. Canny edge detection + morphological closing
5. Quantitative comparison vs ground truth

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BRATS_2D = Path('/kaggle/working/brats2020_2d')

import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import filters, morphology, segmentation, measure
from skimage.segmentation import watershed
from skimage.feature import canny
from scipy import ndimage

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load a slice with visible tumor
with open(BRATS_2D / 'metadata.json') as f:
    meta = json.load(f)
slices = meta['slices']

# Find a slice with >5% tumor coverage for interesting visuals
np.random.seed(42)
for s in np.random.permutation(slices):
    m = np.load(BRATS_2D / s['patient'] / f"slice_{s['slice']:03d}_mask.npy")
    if (m > 0).mean() > 0.05:
        img = np.load(BRATS_2D / s['patient'] / f"slice_{s['slice']:03d}_image.npy")
        msk = m
        break

print(f'Loaded: patient={s["patient"]}  slice={s["slice"]}')
print(f'Tumor fraction: {(msk > 0).mean():.2%}')

# Work with T1ce (best contrast for enhancing tumor)
t1ce = img[2]

def to_uint8(arr):
    a = arr - arr.min()
    return (a / (a.max() + 1e-8) * 255).astype(np.uint8)

t1ce_u8 = to_uint8(t1ce)

## Metric Helper

In [ ]:
def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    """Binary Dice coefficient."""
    pred = pred.astype(bool)
    gt   = gt.astype(bool)
    intersection = (pred & gt).sum()
    return float(2 * intersection / (pred.sum() + gt.sum() + 1e-8))

def iou_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = pred.astype(bool)
    gt   = gt.astype(bool)
    intersection = (pred & gt).sum()
    union = (pred | gt).sum()
    return float(intersection / (union + 1e-8))

gt_tumor = (msk > 0)  # binary: any tumor vs background
print(f'GT tumor pixels: {gt_tumor.sum():,} / {gt_tumor.size:,}  ({gt_tumor.mean():.2%})')

## 1. Otsu Thresholding

In [ ]:
# Global Otsu
thresh_otsu = filters.threshold_otsu(t1ce)
otsu_binary = t1ce > thresh_otsu

# Remove small noise and skull (keep largest connected component)
otsu_cleaned = morphology.remove_small_objects(otsu_binary, min_size=64)

print(f'Otsu threshold: {thresh_otsu:.4f}')
print(f'Otsu Dice: {dice_score(otsu_cleaned, gt_tumor):.4f}  IoU: {iou_score(otsu_cleaned, gt_tumor):.4f}')

# Multi-level Otsu (3 classes)
thresholds = filters.threshold_multiotsu(t1ce, classes=3)
regions = np.digitize(t1ce, thresholds)  # 0, 1, 2
# Highest intensity region = most likely enhancing tumor
multi_otsu_pred = (regions == 2)
multi_otsu_pred = morphology.remove_small_objects(multi_otsu_pred, min_size=32)

print(f'Multi-Otsu thresholds: {thresholds}')
print(f'Multi-Otsu (top class) Dice: {dice_score(multi_otsu_pred, gt_tumor):.4f}  IoU: {iou_score(multi_otsu_pred, gt_tumor):.4f}')

In [ ]:
from matplotlib.colors import ListedColormap
MASK_CMAP = ListedColormap([
    (0, 0, 0, 0),
    (0.9, 0.2, 0.2, 0.6),
    (0.2, 0.8, 0.2, 0.6),
    (0.9, 0.9, 0.2, 0.6),
])
BIN_CMAP = ListedColormap([(0,0,0,0), (0.2,0.6,1.0,0.6)])

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(t1ce, cmap='gray'); axes[0].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3); axes[0].set_title('T1ce + GT'); axes[0].axis('off')
axes[1].imshow(t1ce, cmap='gray'); axes[1].imshow(otsu_binary, cmap=BIN_CMAP); axes[1].set_title(f'Otsu (thresh={thresh_otsu:.2f})\nDice={dice_score(otsu_binary, gt_tumor):.3f}'); axes[1].axis('off')
axes[2].imshow(t1ce, cmap='gray'); axes[2].imshow(otsu_cleaned, cmap=BIN_CMAP); axes[2].set_title(f'Otsu + cleaned\nDice={dice_score(otsu_cleaned, gt_tumor):.3f}'); axes[2].axis('off')
axes[3].imshow(t1ce, cmap='gray'); axes[3].imshow(multi_otsu_pred, cmap=BIN_CMAP); axes[3].set_title(f'Multi-Otsu (top class)\nDice={dice_score(multi_otsu_pred, gt_tumor):.3f}'); axes[3].axis('off')

plt.suptitle('Otsu Thresholding on T1ce', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Region Growing

In [ ]:
def region_growing(image: np.ndarray, seed: tuple[int,int], tolerance: float) -> np.ndarray:
    """Simple intensity-based region growing from a seed point."""
    h, w = image.shape
    visited = np.zeros((h, w), dtype=bool)
    result  = np.zeros((h, w), dtype=bool)
    seed_val = image[seed]
    
    stack = [seed]
    visited[seed] = True
    
    while stack:
        r, c = stack.pop()
        if abs(float(image[r, c]) - float(seed_val)) <= tolerance:
            result[r, c] = True
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc]:
                    visited[nr, nc] = True
                    stack.append((nr, nc))
    return result

# Use brightest tumor pixel as seed (would be user-clicked in a real system)
tumor_coords = np.argwhere(gt_tumor)
# Pick the pixel with highest T1ce intensity among GT tumor pixels
intensities = t1ce[gt_tumor]
best_idx = np.argmax(intensities)
seed = tuple(tumor_coords[best_idx])
print(f'Seed pixel: {seed}  T1ce intensity: {t1ce[seed]:.4f}')

# Try different tolerances
results_rg = {}
for tol in [0.1, 0.3, 0.5]:
    rg = region_growing(t1ce, seed, tolerance=tol)
    results_rg[tol] = rg
    print(f'  tolerance={tol:.1f}  pixels={rg.sum():,}  Dice={dice_score(rg, gt_tumor):.4f}  IoU={iou_score(rg, gt_tumor):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(t1ce, cmap='gray'); axes[0].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3)
axes[0].scatter(seed[1], seed[0], c='cyan', s=100, zorder=5, marker='x'); axes[0].set_title('GT + seed'); axes[0].axis('off')

for i, (tol, rg) in enumerate(results_rg.items()):
    axes[i+1].imshow(t1ce, cmap='gray')
    axes[i+1].imshow(rg, cmap=BIN_CMAP)
    axes[i+1].scatter(seed[1], seed[0], c='cyan', s=80, zorder=5, marker='x')
    axes[i+1].set_title(f'Region grow tol={tol}\nDice={dice_score(rg, gt_tumor):.3f}')
    axes[i+1].axis('off')

plt.suptitle('Region Growing from brightest tumor seed', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Watershed Segmentation

In [ ]:
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

# Prepare: Gaussian blur → distance transform → local maxima as markers
blurred = ndimage.gaussian_filter(t1ce, sigma=1.5)

# Threshold to get foreground estimate
fg_thresh = filters.threshold_otsu(blurred)
foreground = blurred > fg_thresh

# Distance transform of foreground
distance = ndimage.distance_transform_edt(foreground)

# Local maxima as seed markers
coords = peak_local_max(distance, min_distance=10, labels=foreground)
mask_markers = np.zeros(distance.shape, dtype=bool)
mask_markers[tuple(coords.T)] = True
markers, _ = ndimage.label(mask_markers)

# Watershed on negative image (flood from high-intensity peaks)
ws_labels = watershed(-blurred, markers, mask=foreground)

print(f'Watershed regions found: {ws_labels.max()}')

# Pick regions overlapping with highest-intensity areas (likely tumor)
region_mean_intensity = {}
for r in range(1, ws_labels.max() + 1):
    region_mask = ws_labels == r
    region_mean_intensity[r] = t1ce[region_mask].mean()

# Top 3 brightest regions = tumor candidate
sorted_regions = sorted(region_mean_intensity.items(), key=lambda x: x[1], reverse=True)
top_regions = {r for r, _ in sorted_regions[:3]}
ws_pred = np.isin(ws_labels, list(top_regions)) & foreground

print(f'Watershed (top 3 regions) Dice: {dice_score(ws_pred, gt_tumor):.4f}  IoU: {iou_score(ws_pred, gt_tumor):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(t1ce, cmap='gray'); axes[0].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3); axes[0].set_title('T1ce + GT'); axes[0].axis('off')
axes[1].imshow(distance, cmap='hot'); axes[1].scatter(coords[:, 1], coords[:, 0], c='cyan', s=10); axes[1].set_title(f'Distance transform + seeds ({len(coords)})'); axes[1].axis('off')
axes[2].imshow(segmentation.mark_boundaries(t1ce / (t1ce.max()+1e-8), ws_labels, color=(0.2,0.8,0.2), mode='thick')); axes[2].set_title(f'Watershed boundaries\n({ws_labels.max()} regions)'); axes[2].axis('off')
axes[3].imshow(t1ce, cmap='gray'); axes[3].imshow(ws_pred, cmap=BIN_CMAP); axes[3].set_title(f'Watershed top-3 regions\nDice={dice_score(ws_pred, gt_tumor):.3f}'); axes[3].axis('off')

plt.suptitle('Watershed Segmentation on T1ce', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Canny Edge Detection + Morphological Closing

In [ ]:
# Canny edges
edges_loose = canny(t1ce, sigma=1.0, low_threshold=0.1, high_threshold=0.3)
edges_tight = canny(t1ce, sigma=2.0, low_threshold=0.2, high_threshold=0.5)

# Close edges to form enclosed regions
struct = morphology.disk(3)
closed_loose = ndimage.binary_fill_holes(ndimage.binary_dilation(edges_loose, structure=struct))
closed_tight = ndimage.binary_fill_holes(ndimage.binary_dilation(edges_tight, structure=struct))

# Keep only bright regions (overlap with high T1ce intensity)
bright_mask = t1ce > filters.threshold_otsu(t1ce)
canny_pred_loose = closed_loose & bright_mask
canny_pred_tight = closed_tight & bright_mask

# Remove small components
canny_pred_loose = morphology.remove_small_objects(canny_pred_loose, min_size=32)
canny_pred_tight = morphology.remove_small_objects(canny_pred_tight, min_size=32)

print(f'Canny loose: Dice={dice_score(canny_pred_loose, gt_tumor):.4f}  IoU={iou_score(canny_pred_loose, gt_tumor):.4f}')
print(f'Canny tight: Dice={dice_score(canny_pred_tight, gt_tumor):.4f}  IoU={iou_score(canny_pred_tight, gt_tumor):.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].imshow(t1ce, cmap='gray'); axes[0,0].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3); axes[0,0].set_title('GT overlay'); axes[0,0].axis('off')
axes[0,1].imshow(edges_loose, cmap='gray'); axes[0,1].set_title('Canny edges (σ=1, loose)'); axes[0,1].axis('off')
axes[0,2].imshow(edges_tight, cmap='gray'); axes[0,2].set_title('Canny edges (σ=2, tight)'); axes[0,2].axis('off')

axes[1,0].imshow(t1ce, cmap='gray'); axes[1,0].imshow(msk, cmap=MASK_CMAP, vmin=0, vmax=3); axes[1,0].set_title('GT overlay'); axes[1,0].axis('off')
axes[1,1].imshow(t1ce, cmap='gray'); axes[1,1].imshow(canny_pred_loose, cmap=BIN_CMAP); axes[1,1].set_title(f'Canny+morph (loose)\nDice={dice_score(canny_pred_loose, gt_tumor):.3f}'); axes[1,1].axis('off')
axes[1,2].imshow(t1ce, cmap='gray'); axes[1,2].imshow(canny_pred_tight, cmap=BIN_CMAP); axes[1,2].set_title(f'Canny+morph (tight)\nDice={dice_score(canny_pred_tight, gt_tumor):.3f}'); axes[1,2].axis('off')

plt.suptitle('Canny Edge Detection + Morphological Closing', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Quantitative Comparison

In [ ]:
results = [
    ('Otsu (global)',        otsu_binary),
    ('Otsu + cleaned',       otsu_cleaned),
    ('Multi-Otsu top class', multi_otsu_pred),
    ('Region grow (tol=0.1)', results_rg[0.1]),
    ('Region grow (tol=0.3)', results_rg[0.3]),
    ('Region grow (tol=0.5)', results_rg[0.5]),
    ('Watershed top-3',      ws_pred),
    ('Canny+morph (loose)',  canny_pred_loose),
    ('Canny+morph (tight)',  canny_pred_tight),
]

print(f'{"Method":<30} {"Dice":>8} {"IoU":>8} {"Precision":>10} {"Recall":>8}')
print('-' * 68)
for name, pred in results:
    tp = (pred & gt_tumor).sum()
    fp = (pred & ~gt_tumor).sum()
    fn = (~pred & gt_tumor).sum()
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    print(f'{name:<30} {dice_score(pred, gt_tumor):>8.4f} {iou_score(pred, gt_tumor):>8.4f} {prec:>10.4f} {rec:>8.4f}')

In [ ]:
# Bar chart comparison
names = [r[0] for r in results]
dices = [dice_score(r[1], gt_tumor) for r in results]
ious  = [iou_score(r[1], gt_tumor)  for r in results]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, dices, width, label='Dice', color='steelblue', alpha=0.85)
ax.bar(x + width/2, ious,  width, label='IoU',  color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend()
ax.set_title('Classical segmentation methods — Dice & IoU vs GT tumor mask')
plt.tight_layout()
plt.show()

## Summary

Classical methods give a useful **lower bound** on what the deep model must beat:

| Method | Expected Dice | Key limitation |
|---|---|---|
| Otsu thresholding | 0.3–0.5 | Can't distinguish tumor from skull/artifacts |
| Region growing | 0.4–0.6 | Requires accurate seed; leaks through weak boundaries |
| Watershed | 0.3–0.5 | Over-segments; sensitive to noise |
| Canny + morph | 0.2–0.4 | Fragmented edges; lots of FP |

**U-Net target:** Dice ≥ 0.80 on all tumor classes — far beyond any classical method.

These results are essential for the report's ablation table: they justify why deep learning is necessary.